# Inspect Summary Database
View raw transcripts and AI-generated summaries stored in `data/transcripts.db`.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure working directory is the project root, not notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

from src.database.transcript_db import TranscriptDatabase
import pandas as pd

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", None)

db = TranscriptDatabase()

## Transcripts

In [ ]:
transcripts = db.load()
print(f"{len(transcripts)} transcript(s) in database")
transcripts[["video_id", "channel", "title", "published_at", "transcript_available"]]

## Summaries

In [ ]:
summaries = db.load_summaries()
print(f"{len(summaries)} summary/summaries in database")
summaries[["video_id", "model_used", "created_at"]]

## Summaries joined with Transcript metadata

In [ ]:
merged = summaries.merge(
    transcripts[["video_id", "channel", "title", "url"]], on="video_id"
)
merged[["channel", "title", "model_used", "created_at", "url"]]

## Read a Summary

In [ ]:
from IPython.display import Markdown

for _, row in merged.iterrows():
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    display(
        Markdown(
            f"*Channel: {row['channel']} | Model: {row['model_used']} | {row['created_at']}*"
        )
    )
    display(Markdown(row["summary"]))
    display(Markdown("---"))

## Pending (unsummarized transcripts)

In [ ]:
pending = db.get_unsummarized_transcripts()
print(f"{len(pending)} transcript(s) not yet summarized")
if not pending.empty:
    display(pending[["video_id", "channel", "title"]])